In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import (
    col, when, expr, trim, initcap, lower, upper,
    regexp_extract, regexp_replace, split,
    to_date, lit, coalesce, nanvl, concat_ws, array_remove, array
)
from pyspark.sql.types import DoubleType

In [0]:
def build_issue_type(checks: dict):
    """
    checks: dict of {label: Column expression that is TRUE when there is a problem}
    Returns a Column that concatenates all firing labels with ", ".
    """
    flags = [when(condition, lit(label)).otherwise(lit(None)) for label, condition in checks.items()]
    return concat_ws(", ", *[f for f in flags])



In [0]:
rules_customers = {
    "valid_customer_id": "customer_id IS NOT NULL",
    "valid_customer_name": "customer_name IS NOT NULL AND TRIM(customer_name) != ''",
    "valid_segment": "segment IN ('Home Office','Corporate','Consumer')",
    "valid_region":  "region IN ('North','South','East','West','Central')",
}

# A record fails quarantine if ANY of the above rules is broken
quarantine_rules_customers = "NOT ({})".format(" AND ".join(rules_customers.values()))

@dp.table(name="globalmart_dev.silver.customers_quarantine")
def customers_quarantine():
    return (
        spark.readStream.table("customers_bronze_view")
        .withColumn("is_quarantined", expr(quarantine_rules_customers))
        .filter("is_quarantined = true")
        .withColumn("issue_type", build_issue_type({
            "missing_customer_id":   col("customer_id").isNull(),
            "missing_customer_name": col("customer_name").isNull() | (trim(col("customer_name")) == ""),
            "invalid_segment":       ~col("segment").isin("Consumer", "Corporate", "Home Office"),
            "invalid_region":        ~col("region").isin("North", "South", "East", "West", "Central"),
        }))
        .drop("is_quarantined")
    )

@dp.table(
    name="globalmart_dev.silver.customers_silver",
    comment="Cleaned and deduplicated customers. City/state swap fixed for R4. Segment and region normalised."
)
@dp.expect_all(rules_customers)
def customers_silver():
    df = (
        spark.readStream.table("customers_bronze_view")
        .withColumn("is_quarantined", expr(quarantine_rules_customers))
        .filter("is_quarantined = false")
        .dropDuplicates(["customer_id"])
    )
    df = df.drop(
        "_rescued_data", "_source_file_path", "_source_file_name",
        "_source_modified_time", "_ingested_at", "is_quarantined"
    )
    return df


In [0]:
rules_orders = {
    "valid_order_id":              "order_id IS NOT NULL",
    "valid_customer_id":           "customer_id IS NOT NULL",
    "valid_purchase_date":         "order_purchase_date IS NOT NULL",
    "valid_approved_date":         "order_approved_at IS NOT NULL",
    "valid_carrier_date":          "order_delivered_carrier_date IS NOT NULL",
    "valid_customer_delivery_date":"order_delivered_customer_date IS NOT NULL",
    "valid_order_status":          "order_status IN ('Processing','Shipped','Cancelled','Delivered','Created','Invoiced','Unavailable')",
    "valid_ship_mode":             "ship_mode IN ('Standard Class','Second Class','First Class','Same Day')",
}

# order_estimated_delivery_date intentionally excluded from quarantine —
# R5 is missing it for all rows. It is only warned on, not dropped.
quarantine_rules_orders = "NOT ({})".format(" AND ".join(rules_orders.values()))

@dp.table(name="globalmart_dev.silver.orders_quarantine")
def orders_quarantine():
    return (
        spark.readStream.table("orders_bronze_view")
        .withColumn("is_quarantined", expr(quarantine_rules_orders))
        .filter("is_quarantined = true")
        .withColumn("issue_type", build_issue_type({
            "missing_order_id":       col("order_id").isNull(),
            "missing_customer_id":    col("customer_id").isNull(),
            "missing_purchase_date":  col("order_purchase_date").isNull(),
            "missing_approved_date":  col("order_approved_at").isNull(),
            "missing_carrier_date":   col("order_delivered_carrier_date").isNull(),
            "missing_delivery_date":  col("order_delivered_customer_date").isNull(),
            "invalid_order_status":   ~col("order_status").isin(
                "Processing","Shipped","Cancelled","Delivered","Created","Invoiced","Unavailable"
            ),
            "invalid_ship_mode":      ~col("ship_mode").isin(
                "Standard Class","Second Class","First Class","Same Day"
            ),
        }))
        .drop("is_quarantined")
    )

@dp.table(
    name="globalmart_dev.silver.orders_silver",
    comment="Cleaned orders. Both date formats parsed. R5 estimated_delivery_date null is accepted."
)
@dp.expect_all(rules_orders)
def orders_silver():
    df = (
        spark.readStream.table("orders_bronze_view")
        .withColumn("is_quarantined", expr(quarantine_rules_orders))
        .filter("is_quarantined = false")
        .dropDuplicates(["order_id"])
    )
    df = df.drop(
        "_rescued_data", "_source_file_path", "_source_file_name",
        "_source_modified_time", "_ingested_at", "is_quarantined"
    )
    return df


In [0]:
rules_products = {
    "valid_product_id":   "product_id IS NOT NULL",
    "valid_product_name": "product_name IS NOT NULL AND TRIM(product_name) != ''",
    "valid_category":     "categories IS NOT NULL AND TRIM(categories) != ''",
}

quarantine_rules_products = "NOT ({})".format(" AND ".join(rules_products.values()))

@dp.table(name="globalmart_dev.silver.products_quarantine")
def products_quarantine():
    return (
        spark.readStream.table("globalmart_dev.bronze.products")
        .withColumn("is_quarantined", expr(quarantine_rules_products))
        .filter("is_quarantined = true")
        .withColumn("issue_type", build_issue_type({
            "missing_product_id":   col("product_id").isNull(),
            "missing_product_name": col("product_name").isNull() | (trim(col("product_name")) == ""),
            "missing_category":     col("categories").isNull() | (trim(col("categories")) == ""),
        }))
        .drop("is_quarantined")
    )

@dp.table(
    name="globalmart_dev.silver.products_silver",
    comment="Cleaned products. Category mapped, dates parsed, weight normalised where present, nulls filled safely."
)
@dp.expect_all(rules_products)
def products_silver():
    df = (
        spark.readStream.table("globalmart_dev.bronze.products")
        .withColumn("is_quarantined", expr(quarantine_rules_products))
        .filter("is_quarantined = false")
    )

    # ── Category classification ────────────────────────────────────────────
    # IMPORTANT: Women must be checked before Men — "Women" contains "men"
    # so checking Men first misclassifies every women's item.
    df = df.withColumn("category",
        when(lower(col("categories")).like("%women%"),   "Women")
       .when(lower(col("categories")).like("%men%"),     "Men")
       .when(lower(col("categories")).like("%shoe%"),    "Shoes")
       .when(lower(col("categories")).like("%clothing%"),"Clothing")
       .otherwise("Other")
    )

    # ── Product name cleanup ───────────────────────────────────────────────
    df = df.withColumn("product_name", trim(col("product_name")))

    # ── Date parsing — ISO 8601 with explicit format ───────────────────────
    # Source has "2016-04-01T20:22:54Z". Spark default inference can silently
    # produce null for the Z suffix; explicit format prevents this.
    df = df.withColumn("dateAdded",   to_date(col("dateAdded"),   "yyyy-MM-dd'T'HH:mm:ss'Z'")) \
           .withColumn("dateUpdated", to_date(col("dateUpdated"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))

    # ── Sizes null fill ────────────────────────────────────────────────────
    df = df.withColumn("sizes",
        when(col("sizes").isNull(), "One Size").otherwise(col("sizes"))
    )

    # ── Colors null fill ───────────────────────────────────────────────────
    df = df.withColumn("colors",
        when(col("colors").isNull(), "Not Specified").otherwise(col("colors"))
    )

    # ── Weight processing ─────────────────────────────────────────────────
    # weight is ~100% null in the source. Guard every step with isNotNull()
    # so we never call regexp_extract on a null column.
    # If weight is null, we fill with 0.0 at the end and skip extraction.
    df = df.withColumn("weight_value",
        when(
            col("weight").isNotNull(),
            regexp_extract(col("weight"), r"([0-9.]+)", 1).cast(DoubleType())
        ).otherwise(lit(None).cast(DoubleType()))
    ).withColumn("weight_unit",
        when(
            col("weight").isNotNull(),
            lower(regexp_extract(col("weight"), r"([a-zA-Z]+)", 1))
        ).otherwise(lit(None))
    ).withColumn("weight_kg",
        when(col("weight_unit") == "kg",   col("weight_value"))
       .when(col("weight_unit") == "g",    col("weight_value") / 1000)
       .when(col("weight_unit").isin("lb","lbs","pound","pounds"), col("weight_value") * 0.453592)
       .otherwise(lit(None).cast(DoubleType()))
    )

    # ── Null fills for sparse columns ─────────────────────────────────────
    df = df.withColumn("weight_kg",     when(col("weight_kg").isNull(),     lit(0.0)).otherwise(col("weight_kg"))) \
           .withColumn("manufacturer",  when(col("manufacturer").isNull(),  lit("Unknown")).otherwise(col("manufacturer"))) \
           .withColumn("dimension",     when(col("dimension").isNull(),     lit("Unknown")).otherwise(col("dimension")))

    # ── Rename and drop staging columns ───────────────────────────────────
    df = df.drop("weight", "weight_value", "weight_unit") \
           .withColumnRenamed("weight_kg", "weight")

    df = df.dropDuplicates(["product_id"])

    df = df.drop(
        "_rescued_data", "_source_file_path", "_source_file_name",
        "_source_modified_time", "_ingested_at", "is_quarantined"
    )
    return df



In [0]:
rules_returns = {
    "valid_order_id":     "order_id IS NOT NULL",
    "valid_return_date":  "return_date IS NOT NULL",
    "valid_refund_amount":"refund_amount IS NOT NULL AND refund_amount > 0",
    "valid_return_reason":"return_reason IS NOT NULL",
}

quarantine_rules_returns = "NOT ({})".format(" AND ".join(rules_returns.values()))

@dp.table(name="globalmart_dev.silver.returns_quarantine")
def returns_quarantine():
    return (
        spark.readStream.table("returns_bronze_view")
        .withColumn("is_quarantined", expr(quarantine_rules_returns))
        .filter("is_quarantined = true")
        .withColumn("issue_type", build_issue_type({
            "missing_order_id":      col("order_id").isNull(),
            "missing_return_date":   col("return_date").isNull(),
            "missing_refund_amount": col("refund_amount").isNull() | (col("refund_amount") <= 0),
            "missing_return_reason": col("return_reason").isNull(),
        }))
        .drop("is_quarantined")
    )

@dp.table(
    name="globalmart_dev.silver.returns_silver",
    comment="Cleaned returns. Two-schema merge resolved. Rescued refund_amount recovered. NaN→null. Dates parsed."
)
@dp.expect_all(rules_returns)
def returns_silver():
    df = (
        spark.readStream.table("returns_bronze_view")
        .withColumn("is_quarantined", expr(quarantine_rules_returns))
        .filter("is_quarantined = false")
    )
    df = df.drop(
        "_rescued_data", "_source_file_path", "_source_file_name",
        "_source_modified_time", "_ingested_at", "is_quarantined"
    )
    return df


In [0]:
rules_transactions = {
    "valid_order_id":       "order_id IS NOT NULL",
    "valid_product_id":     "product_id IS NOT NULL",
    "valid_sales":          "sales IS NOT NULL AND sales > 0",
    "valid_quantity":       "quantity IS NOT NULL AND quantity > 0",
    "valid_discount":       "discount IS NOT NULL AND discount >= 0 AND discount <= 1",
    "valid_payment_type":   "payment_type IS NOT NULL",
    "valid_installments":   "payment_installments IS NOT NULL AND payment_installments > 0",
    # profit IS NOT NULL intentionally removed — R4 has null profit by design.
    # Gold layer handles null profit as zero-contribution rows.
}

quarantine_rules_transactions = "NOT ({})".format(" AND ".join(rules_transactions.values()))

@dp.table(name="globalmart_dev.silver.transactions_quarantine")
def transactions_quarantine():
    return (
        spark.readStream.table("transactions_bronze_view")
        .withColumn("is_quarantined", expr(quarantine_rules_transactions))
        .filter("is_quarantined = true")
        .withColumn("issue_type", build_issue_type({
            "missing_order_id":     col("order_id").isNull(),
            "missing_product_id":   col("product_id").isNull(),
            "invalid_sales":        col("sales").isNull() | (col("sales") <= 0),
            "invalid_quantity":     col("quantity").isNull() | (col("quantity") <= 0),
            "invalid_discount":     col("discount").isNull() | (col("discount") < 0) | (col("discount") > 1),
            "missing_payment_type": col("payment_type").isNull(),
            "invalid_installments": col("payment_installments").isNull() | (col("payment_installments") <= 0),
        }))
        .drop("is_quarantined")
    )


@dp.table(
    name="globalmart_dev.silver.transactions_silver",
    comment="Cleaned transactions. R3 Order_ID/Product_ID recovered from rescued data. R4 discount normalised. R4 null profit accepted."
)
@dp.expect_all(rules_transactions)
def transactions_silver():
    df = (
        spark.readStream.table("transactions_bronze_view")

        # Numeric columns already typed by the Bronze view.
        # Remaining string cleanup applied here for safety.
        .withColumn("sales",               when(col("sales") == "?", lit(None)).otherwise(col("sales")))
        .withColumn("quantity",            when(col("quantity") == "?", lit(None)).otherwise(col("quantity")))
        .withColumn("payment_installments",when(col("payment_installments") == "?", lit(None)).otherwise(col("payment_installments")))
        .withColumn("payment_type",        trim(col("payment_type")))

        .withColumn("is_quarantined", expr(quarantine_rules_transactions))
        .filter("is_quarantined = false")
    )
    df = df.drop(
        "_rescued_data", "_source_file_path", "_source_file_name",
        "_source_modified_time", "_ingested_at", "is_quarantined"
    )
    return df

In [0]:

rules_vendors = {
    "valid_vendor_id":     "vendor_id IS NOT NULL AND TRIM(vendor_id) != ''",
    "valid_vendor_name":   "vendor_name IS NOT NULL AND TRIM(vendor_name) != ''",
    "valid_vendor_id_fmt": "vendor_id RLIKE '^VEN[0-9]+$'",
}

quarantine_rules_vendors = "NOT ({})".format(" AND ".join(rules_vendors.values()))

@dp.table(name="globalmart_dev.silver.vendors_quarantine")
def vendors_quarantine():
    return (
        spark.readStream.table("vendors")
        .withColumn("vendor_id",   trim(col("vendor_id")))
        .withColumn("vendor_name", initcap(trim(col("vendor_name"))))
        .withColumn("is_quarantined", expr(quarantine_rules_vendors))
        .filter("is_quarantined = true")
        .withColumn("issue_type", build_issue_type({
            "missing_vendor_id":   col("vendor_id").isNull() | (trim(col("vendor_id")) == ""),
            "missing_vendor_name": col("vendor_name").isNull() | (trim(col("vendor_name")) == ""),
            "invalid_vendor_id_format": ~col("vendor_id").rlike("^VEN[0-9]+$"),
        }))
        .drop("is_quarantined")
    )


@dp.table(
    name="globalmart_dev.silver.vendors_silver",
    comment="Cleaned vendor reference data."
)
@dp.expect_all(rules_vendors)
def vendors_silver():
    df = (
        spark.readStream.table("vendors")
        .withColumn("vendor_id",   trim(col("vendor_id")))
        .withColumn("vendor_name", initcap(trim(col("vendor_name"))))
        .withColumn("is_quarantined", expr(quarantine_rules_vendors))
        .filter("is_quarantined = false")
        .dropDuplicates(["vendor_id"])
    )
    df = df.drop(
        "_rescued_data", "_source_file_path", "_source_file_name",
        "_source_modified_time", "_ingested_at", "is_quarantined"
    )
    return df